In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import math
import platform

pd.set_option('display.float_format',"{:.2f}".format)

# OS에 따라 다른 폰트 지정
if platform.system() == 'Darwin':   # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':  # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:  # Linux (예: Colab, Ubuntu)
    plt.rcParams['font.family'] = 'NanumGothic'

In [2]:
df = pd.read_csv('../공유/final_cleaned_airbnb.csv')
df.shape

(19861, 60)

In [3]:
def recommend_investor(df, persona, initial_investment, top_n=5):
    # 1. 조건 필터링
    filtered = df[
        (df['property_regulation_type'] == persona['property_regulation_type']) &
        (df['room_type'] == persona['room_type']) &
        (df['accommodates'] >= persona['accommodates_min']) &
        (df['neighbourhood_group_cleansed'] == persona['neighbourhood_group_cleansed'])
    ]
    if filtered.empty:
        print("조건 수정 필요")
        return pd.DataFrame()
    
    # 2. 세그먼트 후보 컬럼
    segment_cols = ['number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d',
                    'number_of_reviews_ly', 'log_reviews', 'recent_activity']
    
    # 3. 세그먼트별 성과 지표 계산
    segment_summary = filtered.groupby(['neighbourhood_group_cleansed', 'room_type']).agg(
        median_price=('price', 'median'),
        median_occupancy_rate=('occupancy_rate', 'median'),
        median_revenue=('estimated_revenue_l365d', 'median'),
        listing_count=('price', 'count'),
        **{col: (col, 'median') for col in segment_cols}
    ).reset_index()
    
    # 4. 블루오션/레드오션 판단
    listing_median = segment_summary['listing_count'].median()
    revenue_median = segment_summary['median_revenue'].median()
    
    def ocean(row):
        if row['listing_count'] >= listing_median and row['median_revenue'] >= revenue_median:
            return '레드오션'
        elif row['listing_count'] < listing_median and row['median_revenue'] >= revenue_median:
            return '블루오션'
        else:
            return '기타'
    
    segment_summary['market_position'] = segment_summary.apply(ocean, axis=1)
    
    # 5. 수익 계산
    segment_summary['monthly_revenue'] = segment_summary['median_price'] * segment_summary['median_occupancy_rate'] * 30
    segment_summary['estimated_cost'] = segment_summary['monthly_revenue'] * 0.25   # 비용 25% 가정
    segment_summary['net_profit'] = segment_summary['monthly_revenue'] - segment_summary['estimated_cost']
    segment_summary['months_to_recover'] = initial_investment / segment_summary['net_profit']
    
    # 6. TOP N 추천
    top_segments = segment_summary.sort_values(by='net_profit', ascending=False).head(top_n)
    
    return top_segments

In [7]:
# 사용자 입력으로 persona 만들기
def create_persona():
    print("=== 투자자 페르소나 입력 ===")
    property_regulation_type = input("운영 형태 입력 (Residential_short_term/Residential_long_term/Hotel): ")
    room_type = input("숙소 타입 입력 (Entire home/apt, Private room, Hotel room 등): ")
    accommodates_min = int(input("최소 수용 인원 입력: "))
    neighbourhood_group_cleansed = input("동네 입력 (Manhattan, Brooklyn, Queens, Bronx, Staten Island): ")
    
    # 딕셔너리로 저장
    persona = {
        'property_regulation_type': property_regulation_type,
        'room_type': room_type,
        'accommodates_min': accommodates_min,
        'neighbourhood_group_cleansed': neighbourhood_group_cleansed
    }
    
    return persona

# 사용 예시
persona = create_persona()
initial_investment = int(input("초기 투자금 입력: "))

recommend_investor(df, persona, initial_investment, top_n=5)

=== 투자자 페르소나 입력 ===


,neighbourhood_group_cleansed,room_type,median_price,median_occupancy_rate,median_revenue,listing_count,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,number_of_reviews_ly,log_reviews,recent_activity,market_position,monthly_revenue,estimated_cost,net_profit,months_to_recover
0,Manhattan,Entire home/apt,224.00,0.00,0.00,5757,1.00,0.00,0.00,0.00,0.69,0.00,레드오션,0.00,0.00,0.00,inf
